# NYC Taxi Trip Duration: Exploratory Data Analysis (EDA)

## Project Overview
This project aims to predict the total ride duration of taxi trips in New York City.

The dataset for this analysis can be found here: [Kaggle NYC Taxi Trip Duration Data](https://www.kaggle.com/competitions/nyc-taxi-trip-duration/data).

Following the project guidelines, this notebook is dedicated entirely to **Exploratory Data Analysis (EDA) and Feature Engineering (no modeling)**.

Our objective here is to elegantly explore the dataset, formulate our logic, and build mathematical features that will help our algorithms understand the physical reality of NYC taxi routing. All subsequent model training will be handled in separate Python scripts[cite: 8].

## Table of Contents
1. Data Loading & Diagnostics
2. Initial Data Exploration
3. Feature Engineering
   - Datetime Extraction
   - Geographic Distance (Haversine & Manhattan)
   - Travel Direction (Bearing)

### Libraries and Configuration
To keep this notebook elegant and professional, all data processing and feature engineering logic has been abstracted into custom Python modules (`data_helper.py` and `feature_engineering.py`).

In [36]:
import pandas as pd

pd.options.display.max_columns = 100
from taxi_trip_duration.manipulation.data_helper import *
from taxi_trip_duration.manipulation.feature_engineering import *

### Loading the Dataset
We combine the `train.csv` and `val.csv` splits into a single dataframe for comprehensive Exploratory Data Analysis.

In [37]:
train, val, _ = load_dataset()
df = pd.concat([train, val], ignore_index=True)
df.sample(5)

Loading datasets from: dataset/split/


,id,vendor_id,pickup_datetime,passenger_count,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,store_and_fwd_flag,trip_duration
739133,id2328467,2,2016-01-07 09:42:54,1,-73.985710,40.752895,-73.973404,40.752197,N,566
193967,id1162256,2,2016-01-27 16:41:11,2,-73.993607,40.721237,-74.002251,40.740009,N,573
233511,id0344945,1,2016-05-20 18:57:35,1,-73.987396,40.737427,-73.994743,40.723362,N,419
136079,id0528524,2,2016-06-09 09:47:45,1,-73.976387,40.775917,-73.973572,40.764862,N,394
622544,id2667510,2,2016-01-29 20:11:27,1,-73.872826,40.773937,-73.974716,40.788525,N,1759


### Initial Data Exploration: Features
Before manipulating the data, we must understand our initial variables:

* **id**: A unique identifier for each trip.
* **vendor_id**: A code indicating the provider associated with the trip record.
* **pickup_datetime**: Date and time when the meter was engaged.
* **dropoff_datetime**: Date and time when the meter was disengaged.
* **passenger_count**: The number of passengers in the vehicle (driver entered value).
* **pickup_longitude / pickup_latitude**: The coordinates where the meter was engaged.
* **dropoff_longitude / dropoff_latitude**: The coordinates where the meter was disengaged.
* **store_and_fwd_flag**: Indicates whether the trip record was held in vehicle memory before sending to the vendor ($Y = $ store and forward; $N = $ not a store and forward trip).
* **trip_duration**: Duration of the trip in seconds (Target Variable).

In [38]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1229319 entries, 0 to 1229318
Data columns (total 10 columns):
 #   Column              Non-Null Count    Dtype  
---  ------              --------------    -----  
 0   id                  1229319 non-null  str    
 1   vendor_id           1229319 non-null  int64  
 2   pickup_datetime     1229319 non-null  str    
 3   passenger_count     1229319 non-null  int64  
 4   pickup_longitude    1229319 non-null  float64
 5   pickup_latitude     1229319 non-null  float64
 6   dropoff_longitude   1229319 non-null  float64
 7   dropoff_latitude    1229319 non-null  float64
 8   store_and_fwd_flag  1229319 non-null  str    
 9   trip_duration       1229319 non-null  int64  
dtypes: float64(4), int64(3), str(3)
memory usage: 93.8 MB


In [40]:
checking_data(df)

Data Diagnostics:
 - Shape              : (1229319, 10)
 - Missing Values     : 0
 - Duplicate Records  : 0


## Feature Engineering
To get high performance from our models, we need to invent new features[cite: 7]. This section focuses on expanding the dataset beyond its original dimensions to capture spatial and temporal patterns.

In [41]:
df = date_time(df, eda=True)
df.head()

,id,vendor_id,pickup_datetime,passenger_count,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,store_and_fwd_flag,trip_duration,pickup_hour,pickup_day_of_week,pickup_month,pickup_day_of_year,pickup_date,pickup_time
0,id2793718,2,2016-06-08 07:36:19,1,-73.985611,40.735943,-73.980331,40.760468,N,1040,7,2,6,160,2016-06-08,07:36:19
1,id3485529,2,2016-04-03 12:58:11,1,-73.978394,40.764351,-73.991623,40.749859,N,827,12,6,4,94,2016-04-03,12:58:11
2,id1816614,2,2016-06-05 02:49:13,5,-73.989059,40.744389,-73.973381,40.748692,N,614,2,6,6,157,2016-06-05,02:49:13
3,id1050851,2,2016-05-05 17:18:27,2,-73.990326,40.731136,-73.991264,40.748917,N,867,17,3,5,126,2016-05-05,17:18:27
4,id0140657,1,2016-05-12 17:43:38,4,-73.789497,40.646675,-73.987137,40.759232,N,4967,17,3,5,133,2016-05-12,17:43:38


### Datetime Extraction
Regression models cannot natively interpret raw string timestamps. To allow future models to capture seasonal, weekly, and daily traffic patterns, we extract discrete numerical components from the `pickup_datetime` variable. 

We isolate the following components:
* **Hour of the day** (`pickup_hour`)
* **Day of the week** (`pickup_day_of_week`)
* **Month** (`pickup_month`)
* **Day of the year** (`pickup_day_of_year`)

*Note: Because `eda=True` is passed to the function, we also retain isolated `pickup_date` and `pickup_time` columns to make human-readable visual plotting easier later on.*

In [42]:
df = haversine_distance(df)

### Haversine Distance (Great-Circle Distance)
A fundamental predictor of travel time is the total distance traveled. Since we are given geographical coordinates in degrees, we calculate the spherical distance between the pickup and dropoff points. 

We use the **Haversine formula**, which calculates the shortest path over the earth's surface using an average earth radius of 6371.0 km.
        
Given latitudes $\phi_1, \phi_2$ and longitudes $\lambda_1, \lambda_2$ in radians, the distance $d$ is calculated as:

$$a = \sin^2\left(\frac{\phi_2 - \phi_1}{2}\right) + \cos(\phi_1) \cdot \cos(\phi_2) \cdot \sin^2\left(\frac{\lambda_2 - \lambda_1}{2}\right)$$
$$c = 2 \cdot \arcsin(\sqrt{a})$$
$$d = 6371.0 \cdot c$$

In [43]:
df = manhattan_distance(df)

### Manhattan Distance (L1 Norm)
While Haversine provides the absolute shortest distance, cars in New York City navigate a strict grid system of avenues and streets. 

The **Manhattan-style distance** approximates this grid navigation by calculating the sum of the North-South and East-West distances at the mean latitude.

Where $R$ is the Earth's radius (6371.0 km) and $\phi_{mean}$ is the average latitude:

$$d_{NS} = R \cdot |\phi_2 - \phi_1|$$
$$d_{EW} = R \cdot \cos(\phi_{mean}) \cdot |\lambda_2 - \lambda_1|$$
$$d_{Manhattan} = d_{NS} + d_{EW}$$

In [44]:
df = pickup_bearing(df)

### Pickup Bearing (Direction of Travel)
The direction a taxi is heading can heavily influence traffic density and trip duration. We calculate the initial **Bearing** (compass direction) from the pickup point to the dropoff point. 
        
The resulting angles range from -180 to 180 degrees.

Where $\Delta\lambda$ is the difference in longitude:

$$Y = \sin(\Delta\lambda) \cdot \cos(\phi_2)$$
$$X = \cos(\phi_1) \cdot \sin(\phi_2) - \sin(\phi_1) \cdot \cos(\phi_2) \cdot \cos(\Delta\lambda)$$
$$\theta = \arctan2(Y, X) \cdot \frac{180}{\pi}$$